# Проверочный ноутбук: выкладки статей «Единая геометродинамическая модель» и «Noether–Casimir»
А.Ю. Дроздов, 2026. Ядро: SageMath 9.6.

Структура: каждая **markdown**-ячейка содержит выкладку (как в статье), следующая **code**-ячейка — численную/символьную проверку. Каждая проверка заканчивается `check(...)`, печатающим **OK / FAIL** и величину невязки.

Метки статуса в тексте: **[доказано]** — выкладка строгая; **[численно]** — проверено только численно; **[модель]** — результат зависит от принятой модели.

Ссылки на разделы:
* §1–4 — `theory/Drozdov_unified_theory_detailed_{ru,en}.tex`, раздел «Полный вывод квазимаксвелловской системы».
* §5–9 — разделы о массе, досветовом спине, излучении (`NOTE_mass_of_ring_pair_2026-10-28.md`).
* §10 — `casimir_noether/noether_casimir_drive{,_ru}.tex`, раздел о тензоре натяжений.
* §11–13 — нелинейность, Васильев, Чурляев (`NOTE_Euler_nonlinear_and_field_momentum_2026-10-29.md`).

In [ ]:
# Отключаем препарсер Sage: дальше чистый Python (numpy). Так код идентичен в Sage и в обычном Python.
try:
    preparser(False)
except NameError:
    pass
import numpy as np, itertools
np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(2026)
RESULTS = []
def check(name, err, tol):
    ok = bool(abs(err) < tol)
    RESULTS.append((name, ok, float(abs(err))))
    print(('OK   ' if ok else 'FAIL ') + name + '   |невязка| = %.3e  (допуск %.1e)' % (abs(err), tol))
    return ok
EPS = np.zeros((3,3,3))
for a,b,c in itertools.permutations(range(3)):
    EPS[a,b,c] = np.linalg.det(np.eye(3)[[a,b,c]])
DELTA = np.eye(3)

## 1. Лагранжиан Лазара — скаляр, тензор упругости
**[доказано]** Лагранжиан (Lazar, PLA 374, 3092 (2010), ур. 3)
$$\mathcal L=\tfrac12 p_iv_i+\tfrac12 D_{ij}I_{ij}-\tfrac12\sigma_{ij}\beta_{ij}-\tfrac14 H_{ijk}T_{ijk}.$$
Изотропное определяющее соотношение (ур. 11):
$\sigma_{ij}=\lambda\delta_{ij}\beta_{kk}+\mu(\beta_{ij}+\beta_{ji})+\gamma(\beta_{ij}-\beta_{ji})$.
Утверждение статьи: $\sigma_{ij}=C_{ijkl}\beta_{kl}$ с
$$C_{ijkl}=\lambda\delta_{ij}\delta_{kl}+(\mu+\gamma)\delta_{ik}\delta_{jl}+(\mu-\gamma)\delta_{il}\delta_{jk},$$
и $W=\tfrac12 C_{ijkl}\beta_{ij}\beta_{kl}$ — скаляр (все четыре индекса свёрнуты).

In [ ]:
lam, mu, gam = 1.3, 0.7, 0.4
C = (lam*np.einsum('ij,kl->ijkl',DELTA,DELTA) + (mu+gam)*np.einsum('ik,jl->ijkl',DELTA,DELTA)
     + (mu-gam)*np.einsum('il,jk->ijkl',DELTA,DELTA))
beta = rng.normal(size=(3,3))
sig_formula = lam*np.trace(beta)*DELTA + mu*(beta+beta.T) + gam*(beta-beta.T)
sig_C = np.einsum('ijkl,kl->ij', C, beta)
check('sigma = C beta', np.max(np.abs(sig_formula-sig_C)), 1e-12)
# скалярность: W инвариантна при повороте координат
Q,_ = np.linalg.qr(rng.normal(size=(3,3)))
W  = 0.5*np.einsum('ijkl,ij,kl->', C, beta, beta)
bR = Q@beta@Q.T
WR = 0.5*np.einsum('ijkl,ij,kl->', C, bR, bR)
check('W = 1/2 C beta beta инвариантна при повороте', W-WR, 1e-12)

## 2. Кинематика и тождества Бьянки
**[доказано]** $T_{ijk}=\beta_{ik,j}-\beta_{ij,k}$, $I_{ij}=-v_{i,j}+\dot\beta_{ij}$ (Lazar ур. 1). Тождественно:
$$\epsilon_{jkl}T_{ijk,l}=0,\qquad \dot T_{ijk}+I_{ij,k}-I_{ik,j}=0 .$$
Тензор Ная: $\alpha_{ik}=\epsilon_{klj}\beta_{ij,l}=\tfrac12\epsilon_{klj}T_{ilj}$, вектор Бюргерса $b_i=\int\alpha_{ik}dS_k$.

Проверка — конечными разностями на произвольных гладких полях $\beta(x,t)$, $v(x,t)$.

In [ ]:
def beta_f(x,t):
    return np.array([[np.sin(x[0]+2*x[1]-t+i+3*j)+0.3*np.cos(x[2]*(i+1)+t*j) for j in range(3)] for i in range(3)])
def v_f(x,t):
    return np.array([np.cos(x[0]*(i+1)-x[2]+2*t)+np.sin(x[1]+t*i) for i in range(3)])
h = 1e-4
def dx(f,x,t,k):
    e=np.zeros(3); e[k]=h; return (f(x+e,t)-f(x-e,t))/(2*h)
def dt(f,x,t):
    return (f(x,t+h)-f(x,t-h))/(2*h)
def T_f(x,t):
    dB=[dx(beta_f,x,t,k) for k in range(3)]   # dB[k][i,j] = beta_ij,k
    T=np.zeros((3,3,3))
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i,j,k]=dB[j][i,k]-dB[k][i,j]
    return T
def I_f(x,t):
    dv=[dx(v_f,x,t,j) for j in range(3)]      # dv[j][i] = v_i,j
    return np.array([[-dv[j][i] for j in range(3)] for i in range(3)]) + dt(beta_f,x,t)
x0=np.array([0.3,-0.2,0.5]); t0=0.1
dT=[dx(T_f,x0,t0,l) for l in range(3)]
b1=np.einsum('jkl,lijk->i',EPS,np.array(dT))
check('Бьянки 1: eps_jkl T_ijk,l = 0', np.max(np.abs(b1)), 1e-5)
Tt=dt(T_f,x0,t0); dI=[dx(I_f,x0,t0,k) for k in range(3)]
b2=Tt+np.einsum('kij->ijk',np.array(dI))-np.einsum('jik->ijk',np.array(dI))
check('Бьянки 2: dT/dt + I_ij,k - I_ik,j = 0', np.max(np.abs(b2)), 1e-5)
# тензор Ная
dB=[dx(beta_f,x0,t0,l) for l in range(3)]
alpha1=np.einsum('klj,lij->ik',EPS,np.array(dB))
alpha2=0.5*np.einsum('klj,ilj->ik',EPS,T_f(x0,t0))
check('Най: eps beta_ij,l = 1/2 eps T_ilj', np.max(np.abs(alpha1-alpha2)), 1e-6)

## 3. Сила Пича–Кёлера и тензор Эшелби
**[численно]** В статье: $f_k=\sigma_{ij}T_{ijk}-p_iI_{ik}$ и тождество
$$f_k=-\partial_t(p_i\beta_{ik})+\partial_j\big(\sigma_{ij}\beta_{ik}-\delta_{jk}(W-T_{\rm kin})\big)-\beta_{ik}(\sigma_{ij,j}-\dot p_i),$$
где последний член исчезает на уравнениях движения $\dot p_i=\sigma_{ij,j}$. Само алгебраическое тождество верно при любом знаке, если сохранить остаточный член. Знак фиксируется тем, что только при «$-$» остаток равен $-\beta_{ik}(\sigma_{ij,j}-\dot p_i)$, т.е. пропорционален уравнению движения и исчезает на решениях; при «$+$» в остатке стоит $\sigma_{ij,j}+\dot p_i\neq0$.

In [ ]:
rho=1.1
def sig_of(B): return lam*np.trace(B)*DELTA + mu*(B+B.T) + gam*(B-B.T)
def P_f(x,t): return rho*v_f(x,t)
B=beta_f(x0,t0); S=sig_of(B); p=P_f(x0,t0); T=T_f(x0,t0); I=I_f(x0,t0)
divS=sum(dx(lambda y,s:sig_of(beta_f(y,s))[:,j],x0,t0,j) for j in range(3))
pdot=dt(P_f,x0,t0)
Wf=lambda y,s:0.5*np.sum(sig_of(beta_f(y,s))*beta_f(y,s))
Kf=lambda y,s:0.5*rho*v_f(y,s)@v_f(y,s)
def residual(sgn):
    f=np.einsum('ij,ijk->k',S,T)+sgn*np.einsum('i,ik->k',p,I)
    rhs=np.zeros(3)
    for k in range(3):
        divterm=sum(dx(lambda y,s:(sig_of(beta_f(y,s))[:,j]@beta_f(y,s)[:,k]),x0,t0,j) for j in range(3))
        tterm=dt(lambda y,s:P_f(y,s)@beta_f(y,s)[:,k],x0,t0)
        rhs[k]=divterm+sgn*tterm-dx(Wf,x0,t0,k)-sgn*dx(Kf,x0,t0,k)-B[:,k]@(divS+sgn*pdot)
    return np.max(np.abs(f-rhs))
# Тождество выполняется алгебраически при любом знаке, если сохранить остаточный член.
# Различает знаки то, ЧТО стоит в остаточном члене: при "-" это (sigma_ij,j - pdot_i) — ровно
# уравнение движения, и он исчезает на решениях; при "+" стоит (sigma_ij,j + pdot_i) — не исчезает.
r_minus=residual(-1); r_plus=residual(+1)
check('тождество Эшелби (знак -) с остатком -beta(div sigma - pdot)', r_minus, 1e-5)
check('тождество Эшелби (знак +) с остатком -beta(div sigma + pdot)', r_plus, 1e-5)
print('=> лишь при знаке "-" остаток пропорционален уравнению движения pdot = div sigma и обращается в 0.')

## 4. Неприводимое разложение энергии дислокаций
**[численно]** $W_T=\tfrac14H_{ijk}T_{ijk}$, $H_{ijk}=c_1T_{ijk}+c_2(T_{jki}-T_{kji})+c_3(\delta_{ij}T_{llk}-\delta_{ik}T_{llj})$ (Lazar ур. 13).
Утверждение: на неприводимых частях кручения (тензорная, векторная, аксиальная) энергия диагональна с модулями
$$c_1-c_2,\qquad c_1-c_2+2c_3,\qquad c_1+2c_2,$$
— ровно комбинации из условий устойчивости Лазара (ур. 14). Лагранжиан Васильева («только аксиальное кручение») ⇔ $c_2=c_1$, $c_3=0$.

In [ ]:
def H_of(T,c1,c2,c3):
    tr=np.einsum('llk->k',T)
    return (c1*T + c2*(np.einsum('ijk->jki',T)-np.einsum('ijk->kji',T))
            + c3*(np.einsum('ij,k->ijk',DELTA,tr)-np.einsum('ik,j->ijk',DELTA,tr)))
R=rng.normal(size=(3,3,3)); Tr=R-np.einsum('ijk->ikj',R)
ax=np.einsum('ijk,ijk->',EPS,Tr)/6*EPS
trv=np.einsum('llk->k',Tr)
vec=(np.einsum('ij,k->ijk',DELTA,trv)-np.einsum('ik,j->ijk',DELTA,trv))/2
ten=Tr-ax-vec
expected={'тензорная':(1,-1,0),'векторная':(1,-1,2),'аксиальная':(1,2,0)}
for name,P in [('тензорная',ten),('векторная',vec),('аксиальная',ax)]:
    got=[np.einsum('ijk,ijk->',H_of(P,*c),P)/np.einsum('ijk,ijk->',P,P) for c in [(1,0,0),(0,1,0),(0,0,1)]]
    check('модуль %s части = %s' % (name, expected[name]), np.max(np.abs(np.array(got)-np.array(expected[name]))), 1e-10)
check('перекрёстный член тензорная–аксиальная = 0', np.einsum('ijk,ijk->',H_of(ten,1,1,1),ax), 1e-10)

## 5. Досветовой спин и гиромагнитное отношение
**[доказано]** Классический заряженный шар радиуса $R=r_e/2$ с $\mu_B=e\omega R^2/3c$: $v/c=3\mu_B/(eR)=3/\alpha$.

Пара колец: $S=\hbar/2$ на радиусе $a=\lambdabar_C$ ⇒ $v=S/(ma)=c/2$.
Если бы $S$ держался на $r_e$: $v=c/(2\alpha)$.

Статический диполь $\mu=q_md$ с $q_m\to e$: $g=\mu/((e/2mc)S)=4d/\lambdabar_C$, и $g=2\iff d=a/2$ **[постановка, не вывод]**.

Исправленное замечание: центробежная формула $d_{eq}=J^2/(mA)$ при $J=\hbar/2$, $A=e^2$ даёт $\lambdabar_C/(4\alpha)=a_0/4$ — атомный масштаб, поэтому к осевому зазору она не относится.

In [ ]:
# СГС
alpha=1/137.035999084; c=2.99792458e10; hbar=1.054571817e-27; me=9.1093837015e-28
e=np.sqrt(alpha*hbar*c); re=e**2/(me*c**2); lamC=hbar/(me*c); a0=lamC/alpha; muB=e*hbar/(2*me*c)
check('шар: v/c = 3/alpha', 3*muB/(e*re/2)/(3/alpha)-1, 1e-12)
check('пара: v = c/2 при a = lambdabar_C', (hbar/2)/(me*lamC)/c-0.5, 1e-12)
for d,g_exp in [(lamC/2,2.0),(lamC/4,1.0)]:
    g=e*d/((e/(2*me*c))*(hbar/2))
    check('g = 4d/lambdabar_C при d=%.2f lambdabar_C' % (d/lamC), g-g_exp, 1e-12)
check('d_eq = J^2/(m e^2) = a0/4', ((hbar/2)**2/(me*e**2))/(a0/4)-1, 1e-12)

## 6. Импульс поля движущейся особой точки: парадокс 4/3
**[численно]** Особая точка — монополь поля $T$ (аналог $\mathbf B$). При медленном движении $\mathbf E=-\mathbf v\times\mathbf B/c$, плотность импульса $\mathbf g=\mathbf E\times\mathbf B/4\pi c$. Для оболочки радиуса $r_0$, $U=q^2/2r_0$:
$$p=\tfrac43\,\frac{U v}{c^2}\qquad(\text{как у электрона Абрагама–Лоренца}).$$
Рецепт Ферми (гиперплоскость, ортогональная мировой трубке) возвращает $p=Uv/c^2$.

In [ ]:
q=1.0; r0=1.0; v=1e-3; cc=1.0
N=400; th=(np.arange(N)+0.5)*np.pi/N
r=np.exp(np.linspace(0,np.log(2000),4000)); drr=np.gradient(r)
TH,RR=np.meshgrid(th,r)
B2=(q/RR**2)**2
gz=v*B2*np.sin(TH)**2/(4*np.pi*cc*cc)
p=np.sum(gz*2*np.pi*RR**2*np.sin(TH)*(np.pi/N)*drr[:,None])
U=q**2/(2*r0)
print('p/(Uv/c^2) =', p/(U*v/cc**2))
check('импульс монополя = 4/3 U v / c^2', p/(U*v/cc**2)-4/3, 2e-3)

## 7. Ядро Лазара: конечная энергия особой точки
**[численно]** В градиентной теории Лазара поле монополя регуляризовано:
$T(r)=\dfrac{q}{4\pi r^2}\big[1-e^{-r/\ell}(1+r/\ell)\big]$, $\ell=\sqrt{c_1/\mu}$ (тип Боппа–Подольского).
Утверждение: полевая и градиентная части энергии равны, сумма $=c_1q^2/(8\pi\ell)$ — как оболочка радиуса $\ell$.

In [ ]:
x=np.exp(np.linspace(np.log(1e-6),np.log(1e4),400000)); dxx=np.gradient(x)
Ef=(1-np.exp(-x)*(1+x))/x**2                        # q/(4pi)=1, l=1, c1=1
Uf=0.5*np.sum(Ef**2*4*np.pi*x**2*dxx)
divE=np.gradient(x**2*Ef,x)/x**2
Ub=0.5*np.sum(divE**2*4*np.pi*x**2*dxx)
# c1 q^2/(8 pi l) при q=4pi: 2pi
check('полевая часть = градиентной', Uf/Ub-1, 1e-3)
check('E = c1 q^2/(8 pi l)', (Uf+Ub)/(2*np.pi)-1, 1e-3)

## 8. Излучение пары колец, твист, анаполь
**[численно]** Твист $b(\varphi)=\cos(\varphi/2)$ на кольце: непрерывный заряд $\lambda=\tfrac12\sin(\varphi/2)$ + скачок $-2$ в особой точке; полный заряд 0, дипольный момент $8/3$ (вращается).
Пара в противофазе: поперечные диполи складываются ($16/3$); синфазная: $p_\perp=0$.

Время высвечивания (Лармор), $\omega=c/2a$, $a=\lambdabar_C$, $q^2=\alpha\hbar c$: $\tau=m c^2/P=6\lambdabar_C/(\alpha c)\approx10^{-18}$ с.

Анаполь: для азимутального тока $T=-(a^2/5c)\oint j\,dl$, $|kT|/|p|=(ka)^2/5$; гашение требует $(ka)^2=5$, т.е. $v=\sqrt5\,c$.

In [ ]:
a=1.0; Nphi=200000; phi=(np.arange(Nphi)+0.5)*2*np.pi/Nphi; dphi=2*np.pi/Nphi
lam_q=0.5*np.sin(phi/2)
Q=np.sum(lam_q)*dphi-2
px=np.sum(lam_q*a*np.cos(phi))*dphi-2*a
check('твист: полный заряд = 0', Q, 1e-8)
check('твист: |p| = 8/3', abs(px)-8/3, 1e-8)
def pair_px(shift):
    return px - (np.sum(lam_q*a*np.cos(phi+shift))*dphi-2*a*np.cos(shift))
check('противофаза: |p_perp| = 16/3', abs(pair_px(np.pi))-16/3, 1e-8)
check('синфаза: p_perp = 0', pair_px(0.0), 1e-8)
tau=6*lamC/(alpha*c)
print('tau =', tau, 's')
check('tau ~ 1e-18 с (порядок)', np.log10(tau)+18, 0.5)
w=1/(2*a); J=np.sum(lam_q*a*w*np.array([0*phi+1]).ravel()*0)  # (не используется)
# анаполь
vt=np.stack([-np.sin(phi),np.cos(phi)])*a*w
pos=np.stack([np.cos(phi),np.sin(phi)])*a
p_vec=(pos*lam_q).sum(1)*dphi - 2*a*np.array([1,0])
Jv=(vt*lam_q).sum(1)*dphi - 2*a*w*np.array([0,1])
Tv=-(a**2/5)*Jv
check('анаполь: |kT|/|p| = (ka)^2/5', w*np.linalg.norm(Tv)/np.linalg.norm(p_vec)-(w*a)**2/5, 1e-8)

## 9. Теорема о часах де Бройля
**[доказано]** Носитель инерции $m_0$ на окружности радиуса $a$ со скоростью $v$:
$E=\Gamma m_0c^2$, $J=\Gamma m_0va$, $\omega=v/a$ ⇒ $\dfrac{E}{J\omega}=\dfrac{c^2}{v^2}\ge1$.
Часы де Бройля для мёбиус-пары (период $4\pi$, $\Omega=\omega/2=E/\hbar$, $J=\hbar/2$) требуют $E=J\omega$ ⇒ $v=c$.
При $v=c/2$, $a=\lambdabar_C$: $\Omega=\omega_C/2$ ⇒ $\lambda=2h/p$.

In [ ]:
for beta_v in [0.1,0.5,0.9]:
    G=1/np.sqrt(1-beta_v**2); m0=1.0; aa=1.0; cc=1.0
    ratio=(G*m0*cc**2)/((G*m0*beta_v*aa)*(beta_v/aa))
    check('E/(J omega) = 1/beta^2 при beta=%.1f' % beta_v, ratio-1/beta_v**2, 1e-12)
# гармония фаз: lambda = (h/p) * (omega_C/Omega)
check('v=c/2, a=lambdabar_C  =>  lambda = 2 h/p', (1.0/(0.5/1.0)) - 2.0, 1e-12)

## 10. Казимир: тензор натяжений из полей мод резонатора
**[численно]** Резонатор $a\times b\times h$, TM-мода:
$E_z=A\sin k_xx\sin k_yy\cos k_zz$, $E_{x,y}=-A\frac{k_{x,y}k_z}{K^2}(\dots)\sin k_zz$, $H_{x,y}$ — по статье (гауссовы единицы).
Утверждения статьи:
1. поля удовлетворяют уравнениям Максвелла и граничным условиям идеального проводника;
2. $-\langle T_{zz}\rangle$ (среднее по сечению и периоду) $=\overline w\,k_z^2/k^2$ и **не зависит от $z$**;
3. на стенке давление $=(\langle H_t^2\rangle-\langle E_n^2\rangle)/8\pi$; без члена $-E_n^2$ (как в прежнем Приложении B) получается неверное отношение 1 вместо $k_z^2/k^2$;
4. две пластины с обрезанием $e^{-k/k_m}$: $p_{in}-p_{out}\to-\pi^2/240\cdot\hbar c/a^4$.

In [ ]:
aR,bR,hR=1.0,1.3,2.7; nx,ny,nz=2,1,3; A0=1.0
kx,ky,kz=np.pi*nx/aR,np.pi*ny/bR,np.pi*nz/hR; K2=kx**2+ky**2; k=np.sqrt(K2+kz**2); om=k  # c=1
def TM(x,y,z,t):
    sx,cx,sy,cy,sz,cz=np.sin(kx*x),np.cos(kx*x),np.sin(ky*y),np.cos(ky*y),np.sin(kz*z),np.cos(kz*z)
    E=np.array([-A0*kx*kz/K2*cx*sy*sz, -A0*ky*kz/K2*sx*cy*sz, A0*sx*sy*cz])*np.cos(om*t)
    H=np.array([-A0*k*ky/K2*sx*cy*cz, A0*k*kx/K2*cx*sy*cz, 0*x])*np.sin(om*t)
    return E,H
# 1. Максвелл: rot E = -dH/dt, rot H = dE/dt, div = 0
hh=1e-5; P=np.array([0.31,0.47,0.83]); t=0.2
def comp(F,i,p,t): return F(*p,t)
def grad(fun,idx,p,t):
    g=np.zeros(3)
    for j in range(3):
        e=np.zeros(3); e[j]=hh; g[j]=(fun(*(p+e),t)[idx[0]][idx[1]]-fun(*(p-e),t)[idx[0]][idx[1]])/(2*hh)
    return g
dE=np.array([grad(TM,(0,i),P,t) for i in range(3)]); dH=np.array([grad(TM,(1,i),P,t) for i in range(3)])
rotE=np.array([dE[2,1]-dE[1,2],dE[0,2]-dE[2,0],dE[1,0]-dE[0,1]])
rotH=np.array([dH[2,1]-dH[1,2],dH[0,2]-dH[2,0],dH[1,0]-dH[0,1]])
Et=(TM(*P,t+hh)[0]-TM(*P,t-hh)[0])/(2*hh); Ht=(TM(*P,t+hh)[1]-TM(*P,t-hh)[1])/(2*hh)
check('rot E = -dH/dt', np.max(np.abs(rotE+Ht)), 1e-6)
check('rot H = dE/dt',  np.max(np.abs(rotH-Et)), 1e-6)
check('div E = 0', np.trace(dE), 1e-6)
# граничные условия на z=0: E_x=E_y=0, H_z=0
E0,H0=TM(0.37,0.52,0.0,0.4)
check('BC z=0: E_t = 0', np.max(np.abs(E0[:2])), 1e-12)
# 2. -<T_zz> по сечению и периоду на разных z
X,Y=np.meshgrid((np.arange(80)+0.5)*aR/80,(np.arange(80)+0.5)*bR/80,indexing='ij')
def mTzz(z, without_En=False):
    acc=0
    for tt in (np.arange(16)+0.5)*2*np.pi/om/16:
        E,H=TM(X,Y,z,tt)
        if without_En: Tzz=(-H[0]**2-H[1]**2)/(8*np.pi)
        else: Tzz=(E[2]**2+H[2]**2-E[0]**2-E[1]**2-H[0]**2-H[1]**2)/(8*np.pi)
        acc+=Tzz.mean()
    return -acc/16
wbar=(A0**2/8)*(k**2/K2)/(8*np.pi)
vals=[mTzz(z) for z in [0.0,0.37,1.1,hR]]
print('-<T_zz>(z):', np.array(vals), '   w kz^2/k^2 =', wbar*kz**2/k**2)
check('-<T_zz> = w kz^2/k^2, не зависит от z', np.max(np.abs(np.array(vals)/(wbar*kz**2/k**2)-1)), 1e-6)
# 3. без члена -E_n^2 на стенке
ratio_wrong=mTzz(0.0,True)/wbar; ratio_right=mTzz(0.0)/wbar
print('отношение p/w: с -E_n^2 =', ratio_right, '  без него =', ratio_wrong, '  kz^2/k^2 =', kz**2/k**2)
check('Приложение B: без -E_n^2 получается 1 (ошибка)', ratio_wrong-1, 1e-6)
# 4. две пластины
def plates(kma, a=1.0):
    km=kma/a
    n=np.arange(1,int(60*kma)+1); kzn=np.pi*n/a
    p_in=np.sum(kzn**2*km*np.exp(-kzn/km))/(2*np.pi)/a
    kzs=np.linspace(0,60*km,2000001); dk=kzs[1]-kzs[0]
    p_out=np.sum(kzs**2*km*np.exp(-kzs/km))*dk/(2*np.pi)/np.pi
    return p_in-p_out
for kma in [20,50,100]:
    print('k_m a =',kma,'  (p_in-p_out) a^4/(hbar c) =', plates(kma))
check('две пластины -> -pi^2/240', plates(100)+np.pi**2/240, 5e-5)

## 11. Нелинейность: уравнение движения в координатах Эйлера
**[доказано]** Скорость частицы в описании Эйлера: из $x=\xi-u(\xi,t)$ и $Dx/Dt=0$
$$(I-\nabla u)\,v=\partial_tu .$$
(В `science/.../Vyvod.ipynb` прежде решалось $(\nabla u)v=-\partial_tu$ — исправлено.)

**[численно]** Модель «Альманси + линейный Гук + неразрывность»: продольная плоская волна удовлетворяет уравнению с остатком $\propto A^2$ (первый порядок — уравнение Ламе).

**[доказано]** Одномерно: $c(\varepsilon)^2=c_0^2(1+\varepsilon)^{-3}$ ⇒ $|\beta_{\rm geom}|=3/2$; вторая гармоника $A_2/A_1=\sigma/2$, $\sigma=\beta\varepsilon_0kx$; расстояние образования разрыва $x_s=1/(\beta\varepsilon_0k)$.

**[источник]** Полный коэффициент (ЛЛ т. VII §26, задача): $\beta=\tfrac32+\dfrac{A+3B+C}{\lambda+2\mu}$.

In [ ]:
lamE,muE,rho0=1.0,1.0,1.0; cL=np.sqrt((lamE+2*muE)/rho0)
def long_res(A,kk=1.0):
    w=cL*kk; xg=np.linspace(0,2*np.pi,2001); t=0.3; hd=1e-4
    u=lambda x,t: A*np.cos(kk*x-w*t)
    ux=lambda x,t:(u(x+hd,t)-u(x-hd,t))/(2*hd); ut=lambda x,t:(u(x,t+hd)-u(x,t-hd))/(2*hd)
    vv=lambda x,t: ut(x,t)/(1-ux(x,t)); rr=lambda x,t: rho0*(1-ux(x,t))
    sg=lambda x,t:(lamE+2*muE)*(ux(x,t)-0.5*ux(x,t)**2)
    vt=(vv(xg,t+hd)-vv(xg,t-hd))/(2*hd); vx=(vv(xg+hd,t)-vv(xg-hd,t))/(2*hd); sx=(sg(xg+hd,t)-sg(xg-hd,t))/(2*hd)
    return np.max(np.abs(rr(xg,t)*(vt+vv(xg,t)*vx)-sx))
rs=[long_res(A)/A**2 for A in [1e-2,5e-3,2.5e-3]]
print('остаток/A^2 =', rs)
check('остаток продольной волны ~ A^2', (rs[0]-rs[2])/rs[2], 0.02)
eps_=np.linspace(-0.01,0.01,5); cvals=(1+eps_)**-1.5
check('|beta_geom| = 3/2', abs(np.gradient(cvals,eps_)[2])-1.5, 1e-3)
eps0,kk,bt=1e-3,1.0,1.5
for xx in [0.1,0.6]:
    sgm=bt*eps0*kk*xx; Nt=4096; tau_=np.linspace(0,2*np.pi,Nt,endpoint=False); ph=tau_.copy()
    for _ in range(200): ph=tau_-sgm*np.sin(ph)
    sp=np.abs(np.fft.rfft(eps0*np.sin(ph)))
    check('A2/A1 = sigma/2 (x=%.1f)' % xx, sp[2]/sp[1]/(sgm/2)-1, 1e-4)

## 12. Васильев: суперпозиция в классическом «дираковском» эфире
**[численно]** Спинор $\xi$ → корепер (поворот $R(\xi/|\xi|)$), плотность $\rho=|\xi|^2$.
Полевое уравнение по $\rho$: $L=\|\dot\vartheta\|^2-\|T^{ax}\|^2=0$ (Chervova–Vassiliev 2009, ур. 18, сигнатура $(-,-,-)$).
1. Одна плоская волна Вейля $\xi=\zeta e^{-i\omega(t\mp\mathbf n\cdot\mathbf x)}$: $\|\dot\vartheta\|^2=\|T^{ax}\|^2=\tfrac{16}{9}\omega^2$, $L=0$.
2. Сумма двух волн: $L=0$, если совпадают направления **или** частоты; $L\ne0$, если различны и направления, и частоты — **суперпозиция нарушена**.

Оговорка: проверено только уравнение по $\rho$, не полная система по кореперу.

In [ ]:
SIG=[np.array([[0,1],[1,0]],complex),np.array([[0,-1j],[1j,0]]),np.array([[1,0],[0,-1]],complex)]
def frame(xi):
    n=np.linalg.norm(xi); a_,b_=xi/n
    U=np.array([[a_,-np.conj(b_)],[b_,np.conj(a_)]])
    return np.array([[0.5*np.trace(SIG[i]@U@SIG[j]@U.conj().T).real for j in range(3)] for i in range(3)])
def L_at(xi_f,x,t,hd=1e-5):
    Rm=frame(xi_f(x,t))
    dR=[(frame(xi_f(x+hd*np.eye(3)[b],t))-frame(xi_f(x-hd*np.eye(3)[b],t)))/(2*hd) for b in range(3)]
    dRt=(frame(xi_f(x,t+hd))-frame(xi_f(x,t-hd)))/(2*hd)
    tax=sum(np.einsum('abc,a,bc->',EPS,Rm[:,j],np.array([dR[b][:,j] for b in range(3)])) for j in range(3))/3
    th2=sum(np.outer(Rm[:,j],dRt[:,j])-np.outer(dRt[:,j],Rm[:,j]) for j in range(3))/3
    return 0.5*np.sum(th2**2), tax**2
def eigsp(n):
    Mx=sum(n[i]*SIG[i] for i in range(3)); wv,vv=np.linalg.eigh(Mx); return vv[:,np.argmax(wv)]
def wave(om,n):
    z=eigsp(n); return lambda x,t: z*np.exp(-1j*om*(t-np.dot(n,x)))
pts=[(rng.uniform(-3,3,3),rng.uniform(0,5)) for _ in range(10)]
w1=wave(1.0,np.array([0,0,1.0]))
kin,tx=L_at(w1,*pts[0])
check('одна волна: |thetadot|^2 = 16/9 omega^2', kin-16/9, 1e-6)
check('одна волна: L = 0', max(abs(a_-b_) for a_,b_ in [L_at(w1,x,t) for x,t in pts]), 1e-6)
cases=[('одна ось, разные omega',np.array([0,0,1.0]),1.7,True),
       ('разные оси, одна omega',np.array([0,0.6,0.8]),1.0,True),
       ('разные оси и разные omega',np.array([1.0,0,0]),1.7,False)]
for name,n2,om2,expect_zero in cases:
    w2=wave(om2,n2); sup=lambda x,t,w2=w2: w1(x,t)+0.7*w2(x,t)
    Lmax=max(abs(a_-b_) for a_,b_ in [L_at(sup,x,t) for x,t in pts])
    print(name,': max|L/rho| =',Lmax)
    if expect_zero: check('суперпозиция работает: '+name, Lmax, 1e-5)
    else:           check('суперпозиция НАРУШЕНА: '+name+' (требуем max|L| > 1e-2)', max(0.0, 1e-2-Lmax), 1e-12)

## 13. Чурляев: скорость «закрученной» сферической волны
**[численно]** Лок $\mathbf u=\hat z\,j_0(kr)$ в уравнении Ламе — не точная мода, но в среднем по сфере
$$\rho\omega^2/k^2=(\lambda+4\mu)/3=(c_L^2+2c_T^2)\rho/3 .$$
При $\lambda=-2\mu$ (как у Чурляева): $c_L=0$, $K=\lambda+\tfrac23\mu=-\tfrac43\mu<0$ — среда неустойчива.

In [ ]:
lmb,mmu,kk=1.7,0.6,1.3; hd=1e-4
f=lambda x: np.sin(kk*np.linalg.norm(x))/(kk*np.linalg.norm(x))
def divz(x): ez=np.array([0,0,hd]); return (f(x+ez)-f(x-ez))/(2*hd)
def lame_z(x):
    lap=sum((f(x+hd*np.eye(3)[i])-2*f(x)+f(x-hd*np.eye(3)[i]))/hd**2 for i in range(3))
    ez=np.array([0,0,hd]); return mmu*lap+(lmb+mmu)*(divz(x+ez)-divz(x-ez))/(2*hd)
Rs=2.0; pts_=rng.normal(size=(4000,3)); pts_=Rs*pts_/np.linalg.norm(pts_,axis=1)[:,None]
val=-np.mean([lame_z(p_) for p_ in pts_])/np.mean([f(p_) for p_ in pts_])/kk**2
print('rho omega^2/k^2 =', val, '   (lambda+4mu)/3 =', (lmb+4*mmu)/3)
check('c00^2 = (c_L^2 + 2 c_T^2)/3 (в среднем по сфере)', val/((lmb+4*mmu)/3)-1, 5e-3)
K_bulk=-2*mmu+2*mmu/3; cL2=(-2*mmu+2*mmu)
print('при lambda=-2mu: K =', K_bulk, '  c_L^2*rho =', cL2)
check('lambda = -2mu => c_L = 0', cL2, 1e-12)
check('lambda = -2mu => K = -4/3 mu', K_bulk+4*mmu/3, 1e-12)

## 14. Сонолюминесценция: скачок ε и коэффициенты Боголюбова (Степановский)
**[доказано]** Мода $\ddot\psi+\omega^2(t)\psi=0$, $\omega^2=c^2k^2/\varepsilon(t)$. При мгновенном скачке $\varepsilon\to1$ ($\omega_1/\omega_2=1/\sqrt\varepsilon$):
$$|\beta|^2=\frac{(\omega_2-\omega_1)^2}{4\omega_1\omega_2}=\frac{(\sqrt\varepsilon-1)^2}{4\sqrt\varepsilon}\quad(\text{Степановский, ур. 11}),$$
при $\varepsilon=1.77$: $|\beta|^2=0.0205$ — **потолок**, не зависящий от скорости стенки.

Точная формула для профиля $\omega^2(t)=\tfrac12(\omega_1^2+\omega_2^2)+\tfrac12(\omega_2^2-\omega_1^2)\tanh(t/\tau)$ (Бернар–Данкан):
$$|\beta|^2=\frac{\sinh^2\!\big(\pi\tau(\omega_2-\omega_1)/2\big)}{\sinh(\pi\tau\omega_1)\sinh(\pi\tau\omega_2)} .$$
Проверка: интегрирование RK4 и предел $\tau\to0$; при $\omega\tau\gg1$ — экспоненциальное (адиабатическое) подавление.

In [ ]:
eps_j=1.77
w1,w2=1.0/np.sqrt(eps_j),1.0          # до и после скачка (k=c=1)
n2_sudden=(np.sqrt(eps_j)-1)**2/(4*np.sqrt(eps_j))
check('|beta|^2 (внезапный) = (w2-w1)^2/(4 w1 w2)', n2_sudden-(w2-w1)**2/(4*w1*w2), 1e-14)
check('Степановский: |beta|^2 = 0.0205 при eps=1.77', n2_sudden-0.020515, 5e-6)
def bd(tau): return np.sinh(np.pi*tau*(w2-w1)/2)**2/(np.sinh(np.pi*tau*w1)*np.sinh(np.pi*tau*w2))
def rk4_beta(tau, T=None, n=40000):
    T = T or 40*max(tau,1.0)
    om2=lambda t: 0.5*(w1**2+w2**2)+0.5*(w2**2-w1**2)*np.tanh(t/tau)
    t=-T; x=np.exp(-1j*w1*t)/np.sqrt(2*w1); v=-1j*w1*x; hstep=2*T/n
    f=lambda t,x,v:(v,-om2(t)*x)
    for _ in range(n):
        k1=f(t,x,v); k2=f(t+hstep/2,x+hstep/2*k1[0],v+hstep/2*k1[1])
        k3=f(t+hstep/2,x+hstep/2*k2[0],v+hstep/2*k2[1]); k4=f(t+hstep,x+hstep*k3[0],v+hstep*k3[1])
        x+=hstep/6*(k1[0]+2*k2[0]+2*k3[0]+k4[0]); v+=hstep/6*(k1[1]+2*k2[1]+2*k3[1]+k4[1]); t+=hstep
    return (abs(v)**2+w2**2*abs(x)**2)/(2*w2)-0.5
for tau in [0.3,1.0,3.0]:
    b_num=rk4_beta(tau); b_th=bd(tau)
    print('tau=%.1f  RK4 |beta|^2=%.6e   формула=%.6e' % (tau,b_num,b_th))
    check('Бернар–Данкан = RK4 (tau=%.1f)' % tau, b_num/b_th-1, 1e-3)
check('предел tau->0 = внезапный скачок', bd(1e-5)/n2_sudden-1, 1e-6)
print('адиабатика: |beta|^2(tau=10)/|beta|^2(0) =', bd(10.0)/n2_sudden)
check('omega*tau >> 1: подавление < 1e-3', max(0.0, bd(10.0)/n2_sudden-1e-3), 1e-12)

## 15. Дипольный момент твиста — символьно
**[доказано]** Непрерывный магнитный заряд $\lambda(\varphi)=-db/d\varphi=\tfrac12\sin(\varphi/2)$ на кольце радиуса $a$, скачок $-2$ в $\varphi=0$:
$$p_x=a\!\int_0^{2\pi}\!\tfrac12\sin\tfrac\varphi2\cos\varphi\,d\varphi-2a=\tfrac a4\Big(\tfrac43-4\Big)-2a=-\tfrac{8}{3}a .$$
(численная версия — §8; здесь точный интеграл в Sage, если ядро Sage доступно).

In [ ]:
try:
    SR; HAVE_SAGE=True
except NameError:
    HAVE_SAGE=False
print('Sage доступен:', HAVE_SAGE)
if HAVE_SAGE:
    ph=var('ph')
    px_sym=integrate(sin(ph/2)/2*cos(ph),ph,0,2*pi)-2
    print('p_x/a =', px_sym)
    check('символьно: p_x = -8/3 a', float(px_sym+Integer(8)/3), 1e-14)
    q_sym=integrate(sin(ph/2)/2,ph,0,2*pi)-2
    check('символьно: полный заряд твиста = 0', float(q_sym), 1e-14)
else:
    print('пропущено: нужен Sage (численная проверка — §8)')

## 16. Символьные проверки (Sage)
**[доказано]** Те же утверждения, что проверялись численно, — точными выкладками:
1. $\sigma_{ij}=C_{ijkl}\beta_{kl}$ для изотропного $C$ (произвольная символьная матрица $\beta$).
2. Тождества Бьянки для произвольных функций $\beta_{ij}(x,t)$, $v_i(x,t)$.
3. $-\partial_h\big(\tfrac{\hbar c}{2}k\big)=\dfrac{\hbar c}{2h}\dfrac{k_z^2}{k}$ при $k_z=\pi n_z/h$ (мода Казимира, гамильтонов вывод = тензорный).
4. $\langle T_{zz}\rangle$ TM-моды по сечению и периоду: $8\pi\langle T_{zz}\rangle=-\tfrac{A^2}{8}\,k_z^2/K^2$, без $z$.
5. Ряд $(1+\varepsilon)^{-3/2}=1-\tfrac32\varepsilon+\dots$ ($\beta_{\rm geom}$) и коэффициент ЛЛ $\beta=\tfrac32+(A+3B+C)/(\lambda+2\mu)$ из энергии ЛЛ §26.
6. Предел точной формулы Бернара–Данкана при $\tau\to0$.
7. $E/(J\omega)=c^2/v^2$.

In [ ]:
if HAVE_SAGE:
    # 1
    l_,m_,g_=var('l_ m_ g_')
    Bs=matrix(SR,3,3,lambda i,j: var('b%d%d'%(i,j)))
    D=identity_matrix(SR,3)
    sig1=l_*Bs.trace()*D+m_*(Bs+Bs.transpose())+g_*(Bs-Bs.transpose())
    sig2=matrix(SR,3,3,lambda i,j: sum((l_*D[i,j]*D[k,l]+(m_+g_)*D[i,k]*D[j,l]+(m_-g_)*D[i,l]*D[j,k])*Bs[k,l]
                                      for k in range(3) for l in range(3)))
    check('символьно: sigma = C beta', float(sum(abs((sig1-sig2)[i,j].expand()) for i in range(3) for j in range(3))), 1e-30)
    # 2
    X=var('x0 x1 x2'); tt=var('tt')
    Bf=[[function('B%d%d'%(i,j))(*X,tt) for j in range(3)] for i in range(3)]
    Vf=[function('V%d'%i)(*X,tt) for i in range(3)]
    Tsym=lambda i,j,k: diff(Bf[i][k],X[j])-diff(Bf[i][j],X[k])
    Isym=lambda i,j: -diff(Vf[i],X[j])+diff(Bf[i][j],tt)
    from itertools import permutations
    e3=lambda a,b,c: Permutation([a+1,b+1,c+1]).sign() if len({a,b,c})==3 else 0
    r1=sum(abs(bool(sum(e3(j,k,l)*diff(Tsym(i,j,k),X[l]) for j in range(3) for k in range(3) for l in range(3)).expand()!=0)) for i in range(3))
    r2=sum(abs(bool((diff(Tsym(i,j,k),tt)+diff(Isym(i,j),X[k])-diff(Isym(i,k),X[j])).expand()!=0)) for i in range(3) for j in range(3) for k in range(3))
    check('символьно: Бьянки 1', r1, 0.5); check('символьно: Бьянки 2', r2, 0.5)
    # 3
    hb,cc_,hh_,kx_,ky_,nz_=var('hb cc_ hh_ kx_ ky_ nz_'); assume(hh_>0)
    kk=sqrt(kx_**2+ky_**2+(pi*nz_/hh_)**2)
    lhs=-diff(hb*cc_*kk/2,hh_); rhs=hb*cc_/(2*hh_)*(pi*nz_/hh_)**2/kk
    check('символьно: -d/dh(hbar c k/2) = hbar c kz^2/(2 h k)', float(bool((lhs-rhs).simplify_full()!=0)), 0.5)
    # 4
    A_,Kx,Ky,Kz,xx,yy,zz,a_,b_=var('A_ Kx Ky Kz xx yy zz a_ b_')
    K2=Kx**2+Ky**2; k_=sqrt(K2+Kz**2)
    Ez=A_*sin(Kx*xx)*sin(Ky*yy)*cos(Kz*zz); Ex=-A_*Kx*Kz/K2*cos(Kx*xx)*sin(Ky*yy)*sin(Kz*zz); Ey=-A_*Ky*Kz/K2*sin(Kx*xx)*cos(Ky*yy)*sin(Kz*zz)
    Hx=-A_*k_*Ky/K2*sin(Kx*xx)*cos(Ky*yy)*cos(Kz*zz); Hy=A_*k_*Kx/K2*cos(Kx*xx)*sin(Ky*yy)*cos(Kz*zz)
    # среднее по периоду: cos^2 и sin^2 по t дают 1/2; среднее по сечению при Kx=pi*n/a и т.д. через подстановку
    nX,nY=var('nX nY'); assume(nX,'integer'); assume(nY,'integer'); assume(nX>0); assume(nY>0); assume(a_>0); assume(b_>0)
    avg=lambda F: (integrate(integrate(F.subs(Kx=pi*nX/a_,Ky=pi*nY/b_),xx,0,a_),yy,0,b_)/(a_*b_)).simplify_full()
    Tzz8pi=(avg(Ez**2)-avg(Ex**2)-avg(Ey**2))/2-(avg(Hx**2)+avg(Hy**2))/2
    target=(-A_**2/8*Kz**2/K2).subs(Kx=pi*nX/a_,Ky=pi*nY/b_)
    check('символьно: 8pi<T_zz> = -A^2/8 kz^2/K^2 (без z)', float(bool((Tzz8pi-target).simplify_full()!=0)), 0.5)
    # 5
    ee=var('ee'); s5=taylor((1+ee)**(-Integer(3)/2),ee,0,1)
    check('символьно: (1+e)^(-3/2) = 1 - 3/2 e', float(bool((s5-(1-Integer(3)/2*ee))!=0)), 0.5)
    s,Ka,Mu,AA,BB,CC=var('s Ka Mu AA BB CC')
    U=matrix(SR,3,3,0); U[0,0]=s+s**2/2       # u_xx = s + s^2/2 (ЛЛ 26.1, одномерно)
    Ell=Mu*sum(U[i,j]**2 for i in range(3) for j in range(3))+(Ka/2-Mu/3)*U.trace()**2 \
        +AA/3*sum(U[i,k]*U[i,l]*U[k,l] for i in range(3) for k in range(3) for l in range(3)) \
        +BB*sum(U[i,k]**2 for i in range(3) for k in range(3))*U.trace()+CC/3*U.trace()**3
    c2=Ell.expand().coefficient(s,2); c3=Ell.expand().coefficient(s,3)
    M_=2*c2; N_=3*c3
    beta_LL=(N_/M_).simplify_full()
    target_b=Integer(3)/2+(AA+3*BB+CC)/(Ka+4*Mu/3)
    print('beta =', beta_LL)
    check('символьно: beta = 3/2 + (A+3B+C)/(lambda+2mu)', float(bool((beta_LL-target_b).simplify_full()!=0)), 0.5)
    # 6
    tau_,W1,W2=var('tau_ W1 W2'); assume(W1>0); assume(W2>W1)
    bdsym=sinh(pi*tau_*(W2-W1)/2)**2/(sinh(pi*tau_*W1)*sinh(pi*tau_*W2))
    lim6=limit(bdsym,tau_=0)
    check('символьно: lim tau->0 = (w2-w1)^2/(4 w1 w2)', float(bool((lim6-(W2-W1)**2/(4*W1*W2)).simplify_full()!=0)), 0.5)
    # 7
    m0,vv,aR_,cL_=var('m0 vv aR_ cL_')
    G=1/sqrt(1-vv**2/cL_**2)
    check('символьно: E/(J omega) = c^2/v^2', float(bool(((G*m0*cL_**2)/((G*m0*vv*aR_)*(vv/aR_))-cL_**2/vv**2).simplify_full()!=0)), 0.5)
else:
    print('пропущено: символьные проверки требуют ядро SageMath 9.6')

## Сводка

In [ ]:
n_ok=sum(1 for _,ok,_ in RESULTS if ok)
print('Пройдено %d из %d проверок' % (n_ok, len(RESULTS)))
for name,ok,err in RESULTS:
    if not ok: print('  FAIL:', name, err)